# Phase 7: Prior Case Retrieval - Encoder-Decoder, Full Fine-tuning

## Task
Prior case retrieval: for each query case, find the earlier cases it cites from a pool of candidate cases. The 7 best candidates are returned per query and scored with Micro-F1@7.

Model: the bidirectional encoder of the pre-trained encoder-decoder; all encoder weights are fine-tuned.

## Input
- Phase-7 data (queries and candidates for train / dev / test); downloaded from Hugging Face if not attached (needs HF_TOKEN).
- Phase-1 files legal_encdec.pt and tok_encdec.json.

## Flow of the notebook (from input to output)
1. Write the model code file.
2. Import libraries and set up the file finder.
3. Load the encoder-decoder and set the training settings.
4. Load the queries and candidate cases of the train, dev and test splits.
5. Compute BM25 and TF-IDF scores between every query and every candidate.
6. Define the Micro-F1@7 metric and the score mixing (z-scores, hubness correction, balancing).
7. Define the fine-tuning methods (LoRMA, adapters, full) with memory-saving blocks.
8. Define how documents become token chunks and padded batches.
9. Define how documents become vectors (mean over tokens and chunks) and cosine similarity scores.
10. Define the training loss: contrastive (each query must pick its cited case among hard negatives), plus distillation if a teacher is given.
11. Define the training loop (cosine learning rate, dev check every epoch, keep the best epoch).
12. Tokenize everything, build positive pairs and hard negatives from BM25 / TF-IDF.
13. Fine-tune the model.
14. Compute and save the model scores for all splits.
15. Choose the model weight, BM25 weight and post-processing on dev and write the 4 best submissions.
16. Zip all outputs.

## Output
- 4 submission zips (dev score and settings in the file name).
- Saved score matrices (a file ending in _scores.npz) and a results JSON.
- One zip with all outputs.

In [1]:
%%writefile ed.py
# Encoder-decoder model code (saved as ed.py)
import torch, torch.nn as nn, torch.nn.functional as F, torch.utils.checkpoint
from tokenizers import Tokenizer
PAD = 0
class Attn(nn.Module):
    # It takes d (model size) and h (number of heads) as input, creates the attention layer and returns nothing.
    def __init__(s, d, h): super().__init__(); s.h = h; s.q = nn.Linear(d, d); s.kv = nn.Linear(d, 2 * d); s.o = nn.Linear(d, d)
    # It takes x (input tensor), mem (encoder output), mask (attention mask) and causal (use causal mask) as input, runs self-attention or cross-attention and returns output vectors.
    def forward(s, x, mem=None, mask=None, causal=False):
        m = x if mem is None else mem; B, T, D = x.shape
        q = s.q(x).view(B, T, s.h, -1).transpose(1, 2); k, v = s.kv(m).view(B, m.size(1), 2, s.h, -1).permute(2, 0, 3, 1, 4)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=causal and mask is None)
        return s.o(y.transpose(1, 2).reshape(B, T, D))
class Block(nn.Module):
    # It takes d (model size), h (number of heads), cross (add cross-attention) and p (dropout rate) as input, creates one Transformer block and returns nothing.
    def __init__(s, d, h, cross=False, p=0.1):
        super().__init__(); s.n1, s.sa, s.cross = nn.LayerNorm(d), Attn(d, h), cross
        if cross: s.nx, s.ca = nn.LayerNorm(d), Attn(d, h)
        s.n2, s.ff, s.dp = nn.LayerNorm(d), nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)), nn.Dropout(p)
    # It takes x (input tensor), mem (encoder output), mm (encoder padding mask), causal (use causal mask) and mask (attention mask) as input, runs one Transformer block and returns output vectors.
    def forward(s, x, mem=None, mm=None, causal=False, mask=None):
        x = x + s.dp(s.sa(s.n1(x), mask=mask, causal=causal))
        if s.cross and mem is not None: x = x + s.dp(s.ca(s.nx(x), mem, mm))
        return x + s.dp(s.ff(s.n2(x)))
class EncDec(nn.Module):
    # It takes V (vocabulary size), d (model size), h (number of heads), ne (encoder layers), nd (decoder layers), L (max length) and p (dropout rate) as input, builds the encoder-decoder model and returns nothing.
    def __init__(s, V=16000, d=640, h=10, ne=6, nd=8, L=1024, p=0.1):
        super().__init__(); s.emb, s.pos = nn.Embedding(V, d), nn.Embedding(L, d)
        s.enc = nn.ModuleList(Block(d, h, p=p) for _ in range(ne)); s.dec = nn.ModuleList(Block(d, h, True, p) for _ in range(nd)); s.ne_, s.nd_ = nn.LayerNorm(d), nn.LayerNorm(d)
        s.apply(lambda m: nn.init.normal_(m.weight, 0, 0.02) if isinstance(m, (nn.Linear, nn.Embedding)) else None)
    # It takes x (input tensor) as input, turns token ids into vectors with positions and returns embedding vectors.
    def embed(s, x): return s.emb(x) + s.pos(torch.arange(x.size(1), device=x.device))
    # It takes src (source token ids) as input, runs the encoder over the input text and returns (encoder output, padding mask).
    def encode(s, src):
        mm = (src != PAD)[:, None, None, :]; x = s.embed(src)
        for b in s.enc: x = b(x, mask=mm)
        return s.ne_(x), mm
    # It takes tgt (target token ids), mem (encoder output) and mm (encoder padding mask) as input, runs the decoder and returns next-token scores.
    def decode(s, tgt, mem=None, mm=None):
        x = s.embed(tgt)
        for b in s.dec: x = b(x, mem, mm, causal=True)
        return F.linear(s.nd_(x), s.emb.weight)
    # It takes src (source token ids) and tgt (target token ids) as input, runs encoder then decoder and returns next-token scores (logits).
    def forward(s, src, tgt): return s.decode(tgt, *s.encode(src))

class ETok:
    pad_id = PAD
    # It takes path (file path) as input, loads the tokenizer and returns nothing.
    def __init__(s, path): s.t = Tokenizer.from_file(path)
    # It takes text as input, converts text into token ids and returns a list of token ids.
    def encode(s, text): return s.t.encode(text).ids
class EDB:
    # It takes path (file path) as input, loads the encoder-decoder and keeps its encoder and returns nothing.
    def __init__(s, path):
        d = torch.load(path, map_location="cpu"); s.model = EncDec(**d["cfg"]); s.model.load_state_dict(d["sd"]); s.d_model, s.ck = d["cfg"]["d"], False
        for p in s.model.dec.parameters(): p.requires_grad_(False)
    # It takes x (token ids) and m (mask) as input, returns the encoder vector of every token.
    def hidden(s, x, m=None):
        mm, h = (x != PAD)[:, None, None, :], s.model.embed(x)
        for b in s.model.enc: h = torch.utils.checkpoint.checkpoint(b, h, None, None, False, mm, use_reentrant=False) if s.ck and s.model.training and torch.is_grad_enabled() else b(h, mask=mm)
        return s.model.ne_(h)

Writing ed.py


In [2]:
# Import libraries and file finder helper
import os, re, json, glob, math, time, zipfile, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, torch.utils.checkpoint
DEV, K, OUT = "cuda" if torch.cuda.is_available() else "cpu", 7, "/kaggle/working" if os.path.isdir("/kaggle/working") else "./out"
os.makedirs(OUT, exist_ok=True)
# It takes *patterns (any arguments) as input, finds the right input file to use and returns a path of the chosen file.
def pick(*patterns):
    for p in patterns:
        h = sorted(x for x in glob.glob(f"/kaggle/input/**/{p}", recursive=True) if os.path.isfile(x) and "trainable" not in x)
        if h: print("using", h[0]); return h[0]
    raise FileNotFoundError(f"none of {patterns} found in /kaggle/input")

In [3]:
# Load encoder-decoder model and set training settings
import ed as E
bb = E.EDB(pick("legal_encdec.pt")); tok = E.ETok(pick("tok_encdec.json"))
pid = tok.pad_id or 0
NAME, TECH = "p7_ed_full", "full"
c = dict(lr=5e-05, epochs=2, hours=4.0, bs=8, chunk=512, kinf=4, ktrain=2, tau=0.05, hard=40, r=16, alpha=32, ar=64, teacher="", beta=0.5, teach=0, seed=0)

using /kaggle/input/datasets/patelpreeet/enc-dec-final-pt/legal_encdec.pt
using /kaggle/input/datasets/patelpreeet/tokenizer-enc-dec-p1/tok_encdec.json


In [4]:
# Load queries and candidate cases
# It takes path (file path) as input, loads queries or candidates from a JSON file and returns a list of (id, text, relevant ids).
def read(path):
    r = json.load(open(path, encoding="utf-8"))
    if isinstance(r, dict): r = [dict(v, id=k) if isinstance(v, dict) else dict(id=k, text=v) for k, v in r.items()]
    print("fields:", list(r[0]))
    return [(str(d["id"]), " ".join(d["text"]) if isinstance(d["text"], list) else d["text"], [str(x) for x in d.get("relevant_candidates", [])]) for d in r]
# It takes split (split name) as input, loads the queries and candidates of one split and returns (queries, candidates).
def load_split(split):
    out = []
    for n in (split.lower() + ".json", split.lower() + "_candidates.json"):
        f = [x for x in glob.glob(f"/kaggle/input/**/{n}", recursive=True) if "phase_7" in x.lower()]
        if not f:
            from huggingface_hub import hf_hub_download, list_repo_files, login
            try:
                from kaggle_secrets import UserSecretsClient; login(UserSecretsClient().get_secret("HF_TOKEN"))
            except Exception: login(os.environ.get("HF_TOKEN"))
            repo = "Exploration-Lab/CS-781-Capstone"; f = [hf_hub_download(repo, x, repo_type="dataset") for x in list_repo_files(repo, repo_type="dataset") if x.lower().endswith("/" + n) and "phase_7" in x.lower()]
        out.append(read(f[0]))
    return out
D = {s: load_split(s) for s in ("Train", "Dev", "Test")}
print({s: (len(q), len(c_)) for s, (q, c_) in D.items()}, "(queries, candidates)")

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

fields: ['text', 'relevant_candidates', 'id']


Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

fields: ['text', 'id']


... (6 outputs left out)

fields: ['text', 'id']


Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

fields: ['text', 'id']
{'Train': (827, 4320), 'Dev': (118, 1023), 'Test': (237, 1727)} (queries, candidates)


In [5]:
# Lexical scores for every query and candidate
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from scipy.sparse import csr_matrix
# It takes qs (queries) and cs (candidates) as input, computes BM25 and TF-IDF scores and returns (BM25 matrix, TF-IDF matrix).
def lexical(qs, cs):
    tv = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=2, max_features=300000).fit(cs)
    tfidf = (tv.transform(qs) @ tv.transform(cs).T).toarray()
    cv = CountVectorizer(min_df=2).fit(cs); C = cv.transform(cs).tocoo(); n = C.shape[0]
    df, dl = np.bincount(C.col, minlength=C.shape[1]), np.bincount(C.row, weights=C.data, minlength=n)
    w = C.data * 2.2 / (C.data + 1.2 * (.25 + .75 * dl[C.row] / dl.mean())) * np.log(1 + (n - df[C.col] + .5) / (df[C.col] + .5))
    Q = cv.transform(qs); Q.data = np.log1p(Q.data)
    return (Q @ csr_matrix((w, (C.row, C.col)), shape=C.shape).T).toarray(), tfidf
Lx = {s: lexical([q[1] for q in qs], [d[1] for d in cs]) for s, (qs, cs) in D.items()}
lex = Lx

In [6]:
# Scoring metric (Micro-F1@7) and score mixing
z = lambda S: (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)
# It takes S (score matrix), qs (queries) and cs (candidates) as input, gets the top 7 candidates for each query and returns indices of the top 7 candidates per query.
def top7(S, qs, cs):
    S = np.where(np.array([d[0] for d in cs])[None] == np.array([q[0] for q in qs])[:, None], -1e9, S); return np.argsort(-S, 1)[:, :K]
# It takes S (score matrix), qs (queries) and cs (candidates) as input, computes Micro-F1 at 7 and returns the Micro-F1@7.
def micro_f1(S, qs, cs):
    tp = sum(len({cs[j][0] for j in r} & set(q[2])) for r, q in zip(top7(S, qs, cs), qs)); p, r = tp / (K * len(qs)), tp / sum(len(q[2]) for q in qs); return 2 * p * r / (p + r + 1e-9)
# It takes S (score matrix), lam (strength) and f (fraction) as input, lowers the scores of candidates that match too many queries and returns a adjusted score matrix.
def hub(S, lam, f):
    return S - lam * np.sort(S, 0)[-max(1, int(f * len(S))):].mean(0)
# It takes S (score matrix), t (temperature) and it (number of rounds) as input, balances scores so each candidate fits few queries and returns a balanced score matrix.
def sink(S, t, it=3):
    P = np.exp((S - S.max()) / t) + 1e-30
    for _ in range(it): P /= P.sum(0, keepdims=True); P /= P.sum(1, keepdims=True)
    return np.log(P)
# It takes model, lex (BM25 and TF-IDF scores), a (model weight), b (BM25 weight) and post (post-processing setting) as input, mixes model, BM25 and TF-IDF scores and returns a mixed score matrix.
def blend(model, lex, a, b, post):
    S = a * z(model) + (1 - a) * (b * z(lex[0]) + (1 - b) * z(lex[1])); return hub(S, *post[1:]) if post[0] == "hub" else sink(S, post[1])

In [7]:
# Fine-tuning methods: LoRMA, adapter, full
class LoRMA(nn.Module):
    # It takes base (base linear layer), r (rank) and alpha (scale of the low-rank update) as input, creates the small LoRMA matrices around a linear layer and returns nothing.
    def __init__(s, base, r, alpha):
        super().__init__(); s.base, s.s = base, alpha / r; s.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(r)); s.B = nn.Parameter(torch.zeros(base.out_features, r))
    # It takes x (input tensor) as input, runs the linear layer with the LoRMA update and returns the output of the updated layer.
    def forward(s, x): y = F.linear(x, s.base.weight.to(x.dtype)); y = y + s.s * F.linear(F.linear(y, s.A.to(x.dtype)), s.B.to(x.dtype)); return y if s.base.bias is None else y + s.base.bias.to(x.dtype)
class Adapter(nn.Module):
    # It takes blk (model block), d (model size) and r (adapter size) as input, creates a small adapter that starts as no change and returns nothing.
    def __init__(s, blk, d, r): super().__init__(); s.blk, s.dn, s.up = blk, nn.Linear(d, r), nn.Linear(r, d); nn.init.zeros_(s.up.weight); nn.init.zeros_(s.up.bias)
    # It takes x (input tensor) as input, runs the block and adds the adapter output and returns output with the adapter added.
    def forward(s, x): x = s.blk(x); return x + s.up(F.gelu(s.dn(x)))
class Ck(nn.Module):
    # It takes b (block) as input, wraps a block to save GPU memory and returns nothing.
    def __init__(s, b): super().__init__(); s.b = b
    # It returns the attention layer of the wrapped block.
    @property
    def attn(s): return s.b.attn
    # It takes *a (any arguments) as input, runs the block while saving memory and returns block output.
    def forward(s, *a): return torch.utils.checkpoint.checkpoint(s.b, *a, use_reentrant=False) if s.training and torch.is_grad_enabled() else s.b(*a)
# It takes bb (pre-trained backbone) and tech (fine-tuning method) as input, prepares the model for the chosen fine-tuning method and returns the trainable weights.
def adapt(bb, tech):
    lm = bb.model
    if tech != "full":
        for p in lm.parameters(): p.requires_grad_(False)
    if tech == "lorma":
        for n in bb.linear_names():
            par = lm; *path, leaf = n.split(".")
            for q in path: par = getattr(par, q)
            setattr(par, leaf, LoRMA(getattr(par, leaf), c["r"], c["alpha"]))
    if tech == "adapter":
        for i, b in enumerate(lm.blocks):
            if type(b).__name__ == "FeedForward": lm.blocks[i] = Adapter(b, bb.d_model, c["ar"])
        for p in lm.norm_f.parameters(): p.requires_grad_(True)
    if hasattr(bb, "ck"): bb.ck = True
    else: lm.blocks = nn.ModuleList(Ck(b) for b in lm.blocks)
    return [p for p in lm.parameters() if p.requires_grad]

In [8]:
# Convert documents to tokens and batches
# It takes texts as input, tokenizes documents, keeping the first part and returns a list of token arrays.
def tokenize(texts):
    n = c["chunk"] * c["kinf"]; return [np.array(tok.encode(t[:n * 6])[:n] or [0], np.int32) for t in texts]
# It takes rows (token lists) as input, pads token arrays and makes a mask and returns (padded ids, mask).
def pad(rows):
    L = torch.tensor([len(r) for r in rows]); x = torch.full((len(rows), int(L.max())), pid, dtype=torch.long)
    for i, r in enumerate(rows): x[i, :len(r)] = torch.as_tensor(r, dtype=torch.long)
    return x.to(DEV), (torch.arange(x.size(1))[None] < L[:, None]).to(DEV)

In [9]:
# Turn documents into vectors and compare them
# It takes docs (token arrays of the documents) and rng (random generator) as input, turns documents into vectors and returns one vector per document.
def doc_vectors(docs, rng=None):
    segs = []
    for i, d in enumerate(docs):
        n = max(1, math.ceil(len(d) / c["chunk"])); js = rng.choice(n, min(c["ktrain"], n), replace=False) if rng is not None else range(n)
        segs += [(i, d[j * c["chunk"]:(j + 1) * c["chunk"]]) for j in js]
    x, m = pad([s for _, s in segs]); ix = torch.tensor([i for i, _ in segs], device=DEV)
    with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
        h = bb.hidden(x, m); w = m[..., None].to(h.dtype); h = ((h * w).sum(1) / w.sum(1).clamp(min=1)).float()
    return torch.zeros(len(docs), h.size(1), device=DEV).index_add(0, ix, h) / torch.bincount(ix, minlength=len(docs))[:, None]
# It takes Xq (query token ids), Xc (candidate token ids) and bs (batch size) as input, returns a query x candidate similarity matrix; it is the cosine similarity between all queries and candidates.
@torch.no_grad()
def scores(Xq, Xc, bs=24):
    bb.model.eval(); vec = lambda X: torch.cat([doc_vectors(X[i:i + bs]).cpu() for i in range(0, len(X), bs)])
    q, k = vec(Xq), vec(Xc); mu = k.mean(0); return (F.normalize(q - mu, dim=-1) @ F.normalize(k - mu, dim=-1).T).numpy()

In [10]:
# Training loss (contrastive + distillation)
# It takes qi (query indices), di (candidate indices) and rng (random generator) as input, returns a loss value; it is the contrastive loss, plus distillation if a teacher is given.
def batch_loss(qi, di, rng):
    B = len(qi); v = doc_vectors([Xq[i] for i in qi] + [Xc[j] for j in di], rng); v = F.normalize(v - v.mean(0), dim=-1); lg = v[:B] @ v[B:].T / c["tau"]
    lg = lg - 1e4 * torch.tensor([[cj in gold[q] and k != i for k, cj in enumerate(di)] for i, q in enumerate(qi)], device=DEV)
    loss = F.cross_entropy(lg, torch.arange(B, device=DEV))
    if T is not None:
        t = torch.tensor(np.array([[T[q, j] for j in di] for q in qi]), device=DEV) / c["tau"] - (lg < -5e3) * 1e4
        loss = (1 - c["beta"]) * loss + c["beta"] * F.kl_div(F.log_softmax(lg, 1), F.softmax(t, 1), reduction="batchmean")
    return loss

In [11]:
# Training loop
# It takes ps (parameters) and evaluate (evaluation function) as input, trains the model and keeps the best epoch and returns training time in seconds.
def fit(ps, evaluate):
    rng = np.random.default_rng(c["seed"]); opt = torch.optim.AdamW(ps, lr=c["lr"], weight_decay=0.01); sc = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    spe = len(pairs) // c["bs"]; total, t0 = c["epochs"] * spe, time.time(); best, bst = evaluate(), {k: p.detach().cpu().clone() for k, p in bb.model.named_parameters() if p.requires_grad}; print(f"   epoch 0 (no fine-tuning) dev {best:.4f}"); bb.model.train()
    for step in range(1, total + 1):
        for g in opt.param_groups: g["lr"] = c["lr"] * min(1, step / 20) * 0.5 * (1 + math.cos(math.pi * step / total))
        b = [pairs[i] for i in rng.choice(len(pairs), c["bs"], replace=False)]; qi = [x[0] for x in b]; di = [x[1] for x in b] + [int(rng.choice(hard[q])) for q in qi]
        opt.zero_grad(set_to_none=True); loss = batch_loss(qi, di, rng); sc.scale(loss).backward(); sc.unscale_(opt); nn.utils.clip_grad_norm_(ps, 1.0); sc.step(opt); sc.update()
        if step % 20 == 0: print(f"step {step}/{total} loss {loss.item():.4f} | {(time.time() - t0) / 60:.1f} min", flush=True)
        if step % spe == 0 or time.time() - t0 > c["hours"] * 3600:
            v = evaluate(); print(f"   dev micro-F1@7 (model + lexical + hubness) {v:.4f}", flush=True); bb.model.train()
            if v > best: best, bst = v, {k: p.detach().cpu().clone() for k, p in bb.model.named_parameters() if p.requires_grad}
            if time.time() - t0 > c["hours"] * 3600: break
    bb.model.load_state_dict(bst, strict=False); return time.time() - t0

In [12]:
# Prepare training pairs and hard negatives
X = {s: (tokenize([q[1] for q in qs]), tokenize([d[1] for d in cs])) for s, (qs, cs) in D.items()}
(qs, cs), (Xq, Xc) = D["Train"], X["Train"]; ix = {d[0]: j for j, d in enumerate(cs)}
gold = [{ix[r] for r in q[2] if r in ix} for q in qs]; pairs = [(i, j) for i, g in enumerate(gold) for j in g]
lx = z(Lx["Train"][0]) + z(Lx["Train"][1]); hard = [[j for j in np.argsort(-lx[i])[:c["hard"] + len(gold[i])] if j not in gold[i]] for i in range(len(qs))]
T = np.load(pick(c["teacher"]))["train_model"] if c["teacher"] else None; assert T is None or T.shape == (len(qs), len(cs)), "teacher scores do not match the training data"
print(f"{len(qs)} train queries, {len(pairs)} positive pairs, distillation: {T is not None}")

827 train queries, 5308 positive pairs, distillation: False


In [13]:
# Fine-tune the model
dq, dc = D["Dev"]; ps = adapt(bb, TECH); bb.model.to(DEV); ntr, nall = sum(p.numel() for p in ps), sum(p.numel() for p in bb.model.parameters())
print(f"{TECH}: {ntr / 1e6:.2f}M trainable of {nall / 1e6:.1f}M parameters")
ref = lambda: micro_f1(blend(scores(*X["Dev"]), lex["Dev"], .5, .5, ("hub", 1, .08)), dq, dc)
f1_before = micro_f1(scores(*X["Dev"]), dq, dc); print(f"dev micro-F1@7 with the Phase-1 representations only (raw): {f1_before:.4f}")
secs = fit(ps, ref)

full: 44.01M trainable of 101.9M parameters
dev micro-F1@7 with the Phase-1 representations only (raw): 0.1568
   epoch 0 (no fine-tuning) dev 0.3969
step 20/1326 loss 7.3996 | 0.4 min
step 40/1326 loss 3.4193 | 0.6 min
step 60/1326 loss 5.0906 | 0.8 min
step 80/1326 loss 2.4903 | 1.1 min
step 100/1326 loss 4.0924 | 1.3 min
step 120/1326 loss 3.0484 | 1.5 min
step 140/1326 loss 4.2504 | 1.8 min
step 160/1326 loss 2.2162 | 2.0 min
step 180/1326 loss 1.7481 | 2.3 min
step 200/1326 loss 3.4254 | 2.6 min
step 220/1326 loss 4.0560 | 2.8 min
step 240/1326 loss 4.9577 | 3.1 min
step 260/1326 loss 1.0130 | 3.3 min
step 280/1326 loss 3.2989 | 3.6 min
step 300/1326 loss 3.0249 | 3.9 min
step 320/1326 loss 7.3044 | 4.1 min
step 340/1326 loss 1.7484 | 4.4 min
step 360/1326 loss 3.4956 | 4.6 min
step 380/1326 loss 1.9868 | 4.9 min
step 400/1326 loss 4.4014 | 5.1 min
step 420/1326 loss 4.7234 | 5.4 min
step 440/1326 loss 1.7279 | 5.6 min
step 460/1326 loss 2.8278 | 5.9 min
step 480/1326 loss 1.3083 

In [14]:
# Compute and save scores for all splits
S = {s: scores(*X[s]) for s in ("Dev", "Test")}
if c["teach"]: S["Train"] = scores(Xq, Xc)
np.savez_compressed(f"{OUT}/{NAME}_scores.npz", **{f"{s.lower()}_model": v for s, v in S.items()}, **{f"{s.lower()}_{k}": Lx[s][i] for s in Lx for i, k in enumerate(("bm25", "tfidf"))},
                    **{f"{s.lower()}_{k}": np.array([d[0] for d in D[s][i]]) for s in D for i, k in enumerate(("qids", "cids"))})

In [15]:
# Pick best settings on dev and write submissions
posts = [("hub", l, k) for l in (0, .5, 1, 1.5, 2, 3) for k in (.02, .08, .2)] + [("sink", t) for t in (.2, .35, .5, 1)]
res = {(a_, b_, p): micro_f1(blend(S["Dev"], lex["Dev"], a_, b_, p), dq, dc) for a_ in (1, .7, .5, .35, .25, .15, .08, 0) for b_ in (.5, 1) for p in posts}
f1_lex = micro_f1(z(lex["Dev"][0]) + z(lex["Dev"][1]), dq, dc); f1_raw = micro_f1(S["Dev"], dq, dc); print(f"dev micro-F1@7 | lexical only {f1_lex:.4f} | model only {f1_raw:.4f} | model before fine-tuning {f1_before:.4f}")
tq, tc = D["Test"]; top = sorted(res, key=res.get, reverse=True); print("best settings:", [(k, round(res[k], 4)) for k in top[:5]])
for a_, b_, p in top[:4]:
    pred = {q[0]: [tc[j][0] for j in r] for q, r in zip(tq, top7(blend(S["Test"], lex["Test"], a_, b_, p), tq, tc))}
    assert len(pred) == len(tq) and all(len(set(v)) == K for v in pred.values())
    f = f"{OUT}/{NAME}_dev{res[(a_, b_, p)]:.4f}_m{a_}_b{b_}_{p[0]}{'_'.join(map(str, p[1:]))}_submission.zip"; zipfile.ZipFile(f, "w", zipfile.ZIP_DEFLATED).writestr("prediction.json", json.dumps(pred)); print(f)
json.dump(dict(name=NAME, technique=TECH, trainable_M=ntr / 1e6, total_M=nall / 1e6, train_seconds=secs, dev_model_only=f1_raw, dev_before_ft=f1_before, dev_lexical=f1_lex, grid={str(k): v for k, v in res.items()}, settings=c), open(f"{OUT}/{NAME}_summary.json", "w"), indent=1)

dev micro-F1@7 | lexical only 0.3180 | model only 0.2246 | model before fine-tuning 0.1568
best settings: [((0.08, 0.5, ('hub', 1, 0.08)), 0.448), ((0, 0.5, ('hub', 1, 0.08)), 0.448), ((0.15, 0.5, ('hub', 1, 0.08)), 0.4469), ((0.08, 0.5, ('sink', 0.5)), 0.4447), ((0, 0.5, ('sink', 0.5)), 0.4436)]
/kaggle/working/p7_ed_full_dev0.4480_m0.08_b0.5_hub1_0.08_submission.zip
/kaggle/working/p7_ed_full_dev0.4480_m0_b0.5_hub1_0.08_submission.zip
/kaggle/working/p7_ed_full_dev0.4469_m0.15_b0.5_hub1_0.08_submission.zip
/kaggle/working/p7_ed_full_dev0.4447_m0.08_b0.5_sink0.5_submission.zip


In [16]:
# Zip all outputs
with zipfile.ZipFile(f"{OUT}/ALL_outputs.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(glob.glob(f"{OUT}/*")):
        if os.path.isfile(p) and not p.endswith("ALL_outputs.zip"): zf.write(p, os.path.basename(p))